<p class="lead">Exploratory data analysis is the stage in which you find out what you actually have, before any model is allowed an opinion. It is cheap, it is fast, and it is where almost every serious error in a project is caught — or missed. This unit runs a systematic EDA on the laboratory's curated ROCK1 compound table: shapes and types, missingness, distributions, pairwise relations, group comparisons, outliers. Along the way it builds the vocabulary of the <strong>grammar of graphics</strong> in the language that implements it natively — <strong>ggplot2</strong> — generates Anscombe's quartet and a Datasaurus-style counter-example in the document itself, redraws four classic bad charts correctly, and ends by documenting three genuine anomalies in the data that no summary statistic reveals.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad enseña análisis exploratorio de datos (AED) y la gramática de gráficos en su lengua materna, <code>ggplot2</code>. Ejecutamos una lista de verificación sistemática sobre la tabla curada de compuestos ROCK1: dimensiones y tipos, mapa de datos faltantes, distribuciones, relaciones por pares, comparación entre grupos y valores atípicos. Presentamos las capas de la gramática (datos → estéticas → geometrías → estadísticos → escalas → facetas); comparamos histograma, densidad y función de distribución empírica (ECDF); mostramos cómo engañan los diagramas de caja, violín y puntos; tratamos el sobretrazado, las matrices de correlación con <code>corrplot</code>, los múltiplos pequeños y las escalas logarítmicas. Generamos el cuarteto de Anscombe (que R trae incorporado) y un ejemplo tipo Datasaurus dentro del propio documento, discutimos el color (viridis, Okabe–Ito y una simulación de daltonismo) y redibujamos cuatro gráficos defectuosos. Terminamos documentando tres anomalías reales del conjunto de datos.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Run a systematic EDA — shapes, types, distributions, missingness map, pairwise relations, group comparisons, outliers — before any modelling.</li>
<li>Choose the right plot for the question (distribution, comparison, relationship, composition) and build it in <code>ggplot2</code> with explicit aesthetics, units and colour-blind-safe palettes.</li>
<li>Explain the grammar of graphics and map any figure onto its layers.</li>
<li>Compare histogram, density estimate and ECDF, and state what each hides; know how bin width and bandwidth change the story.</li>
<li>Recognise and fix the standard visual anti-patterns: truncated axes, dual axes, rainbow colour maps, overplotting, pie charts.</li>
<li>Read a plot critically — what it shows, what it hides, what it cannot show — and demonstrate with Anscombe's quartet and a Datasaurus-style construction why summary statistics are never enough.</li>
<li>Find and document real anomalies in a bioactivity dataset and trace each to its experimental cause.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Read linearly, running each chunk and reading its output before moving on. Green <strong>domain-context boxes</strong> define every chemical term on first use; no chemistry background is assumed. In a fresh Google Colab R runtime, run once in a cell of your own: <code>install.packages(c("tidyverse","patchwork","corrplot","pheatmap","e1071","viridisLite"))</code>. This unit follows Unit 4, which built the table we explore, and feeds Unit 6 (curation and bias) and Unit 23 (publication figures).</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(dplyr); library(tidyr); library(stringr); library(ggplot2)
  library(patchwork); library(corrplot); library(e1071); library(viridisLite)
})
set.seed(0)
options(width = 120, digits = 6)
invisible(Sys.setlocale("LC_CTYPE", "en_US.UTF-8"))

PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 9.5),
                panel.grid.minor = element_blank(),
                axis.line = element_line(colour = "grey40", linewidth = 0.3),
                legend.key.size = unit(0.3, "cm")))

REPO <- "DanielBustosG/DanielBustosG.github.io"
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO,
                   "/main/DataScience/data/")
data_path <- function(name) {
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE)
  }
  dest
}
load_data <- function(name, ...)
  read.csv(data_path(name), stringsAsFactors = FALSE, na.strings = c("", "NA"), ...)

cat("R", as.character(getRversion()), "| ggplot2", as.character(packageVersion("ggplot2")),
    "| dplyr", as.character(packageVersion("dplyr")),
    "| corrplot", as.character(packageVersion("corrplot")),
    "| e1071", as.character(packageVersion("e1071")), "\n")



## 1. What exploratory data analysis is for

Tukey's argument, now fifty years old, is that data analysis has two modes: *confirmatory*, which tests a hypothesis you had before you saw the data, and *exploratory*, which finds the hypotheses worth testing. Mixing them — looking at the data, finding a pattern, then testing that pattern on the same data — produces the most common kind of false discovery in science. The discipline is therefore to explore deliberately and openly, and to treat everything found by exploration as a hypothesis rather than a result.

A practical EDA follows the same checklist every time, and the value of a checklist is that it catches the thing you were not looking for.

1. **Shape and types.** How many rows and columns, what is each column, what is the grain.
2. **Missingness.** Which values are absent, and do they go missing together?
3. **Univariate distributions.** For each variable: centre, spread, shape, bounds, impossible values, pile-ups.
4. **Pairwise relations.** Which variables move together, and is the relation linear, monotone or neither?
5. **Group comparisons.** Does the variable of interest differ by a grouping variable (assay, source, class)?
6. **Outliers and anomalies.** Not "points to delete" but "points to explain".
7. **Write down what you found**, including the things you cannot explain.

We run this checklist on the curated ROCK1 compound table — the output of the pipeline built in Unit 4 — letting each step motivate the plot type it needs. Everything below uses base R and the tidyverse;<sup class="c"><a href="#r36">[36]</a></sup> the Python edition of this unit does the same work with NumPy,<sup class="c"><a href="#r4">[4]</a></sup> SciPy,<sup class="c"><a href="#r5">[5]</a></sup> pandas,<sup class="c"><a href="#r6">[6]</a></sup> matplotlib<sup class="c"><a href="#r2">[2]</a></sup> and seaborn.<sup class="c"><a href="#r3">[3]</a></sup> Ten rules for figures that do their job are worth reading once before you start.<sup class="c"><a href="#r8">[8]</a></sup>

<div class="box bridge"><h4>Domain context — what one row of this table is</h4>
<p>Each row is one <strong>compound</strong>: a small organic molecule identified by its <strong>InChIKey</strong>, a hash of its standardised structure. <code>pActivity</code> is its potency against the human kinase <strong>ROCK1</strong> on a logarithmic scale, −log<sub>10</sub> of the concentration in mol/L that halves the enzyme's activity; 9 means one nanomolar and very potent, 5 means ten micromolar and essentially inactive. <code>censored = TRUE</code> marks a compound whose every measurement was an inequality ("&gt; 10 µM") because it was too weak to measure. The remaining numeric columns are <strong>molecular descriptors</strong> computed from the structure: molecular weight (<code>MolWt</code>), the calculated oil/water partition coefficient <code>MolLogP</code> (a measure of greasiness), the topological polar surface area <code>TPSA</code>, counts of hydrogen-bond donors and acceptors (<code>HBD</code>, <code>HBA</code>) and rotatable bonds (<code>RotB</code>), rings, heavy atoms, the fraction of saturated carbons (<code>FractionCSP3</code>) and <code>QED</code>, a composite "drug-likeness" score between 0 and 1.<sup class="c"><a href="#r32">[32]</a></sup> <code>scaffold_smiles</code> is the compound's <strong>Bemis–Murcko scaffold</strong>: the molecule stripped of its side chains, leaving the ring system and the linkers between rings, which is how medicinal chemists group molecules into families.<sup class="c"><a href="#r35">[35]</a></sup> The data are a frozen snapshot of ChEMBL release 37.<sup class="c"><a href="#r29">[29]</a></sup> The descriptors were computed once with RDKit and ship as columns of the file, because R has no RDKit binding here; this unit only reads them.</p></div>



In [ ]:
cmpd <- load_data("rock1_chembl_curated.csv") |>
  mutate(censored = censored == "True",          # pandas wrote the booleans as text
         wide_spread = wide_spread == "True")
cat("dimensions:", nrow(cmpd), "x", ncol(cmpd),
    "| grain: one row per InChIKey, unique:", !any(duplicated(cmpd$inchikey)), "\n")
num <- names(cmpd)[vapply(cmpd, is.numeric, logical(1))]
cat("\nnumeric columns (", length(num), "): ", paste(num, collapse = ", "), "\n", sep = "")
cat("non-numeric:", paste(setdiff(names(cmpd), num), collapse = ", "), "\n")
miss <- sort(colMeans(is.na(cmpd)), decreasing = TRUE)
cat("\ncolumns with missing values (% of 4,280 rows):\n")
print(round(100 * miss[miss > 0], 1))
const <- names(cmpd)[vapply(cmpd, function(x) length(unique(x)) == 1, logical(1))]
cat("\nconstant columns:", if (length(const)) paste(const, collapse = ", ") else "none", "\n")
knitr::kable(summary(cmpd[c("pActivity", "MolWt", "MolLogP", "TPSA", "QED", "n_measurements")]),
             caption = "Five-number summaries of the main numeric columns")



<p class="caption"><b>Table (Section 1).</b> The five-number summary of the main numeric columns. <code>pActivity</code> spans 3.40–9.82 with a median of exactly 6.00 — a suspiciously round median, and the first hint of the pile-ups examined in Section 11; <code>MolWt</code> runs from 133 to 990 daltons with a median of 391, the size range of a typical drug; <code>QED</code> is centred at 0.54. <code>n_measurements</code> has a median of 1 and a maximum of 24 — most compounds were measured once. Three columns carry missing values, and their pattern is informative rather than accidental: <code>max_phase</code> (94.2% missing) and <code>pref_name</code> (93.1%) are absent for the compounds that were never named or never entered clinical development, and <code>pActivity_sd</code> (89.4%) is missing for every compound with a single measurement, since one value has no standard deviation.</p>

Note the first line of the chunk. The curated file was written by pandas, so its boolean columns arrive in R as the character strings `"True"` and `"False"`; converting them explicitly is the price of a cross-language dataset and the kind of thing an EDA exists to catch. Missingness is the second checklist item and deserves a picture, because what matters is whether the gaps co-occur.



In [ ]:
gap_cols <- names(cmpd)[vapply(cmpd, function(x) any(is.na(x)), logical(1))]
ord <- order(cmpd$n_measurements, cmpd$max_phase, na.last = TRUE)
map_df <- as.data.frame(is.na(cmpd[ord, gap_cols])) |>
  mutate(row = row_number()) |>
  pivot_longer(-row, names_to = "column", values_to = "is_missing") |>
  mutate(column = factor(column, levels = gap_cols))
p1 <- ggplot(map_df, aes(column, row, fill = is_missing)) +
  geom_raster() +
  scale_fill_manual(values = c(`TRUE` = "grey15", `FALSE` = "grey92"), guide = "none") +
  scale_y_reverse(expand = c(0, 0)) +
  labs(x = NULL, y = "compound (sorted by n_measurements)",
       title = "Missingness map (dark = missing)")
LAB <- c("000" = "nothing missing", "111" = "all three", "110" = "name + phase",
         "001" = "sd only", "011" = "phase + sd", "010" = "phase only",
         "100" = "name only", "101" = "name + sd")
combo <- apply(is.na(cmpd[gap_cols]), 1, function(r) paste(as.integer(r), collapse = "")) |>
  table() |> sort(decreasing = TRUE)
cdf <- tibble::tibble(pattern = names(combo), n = as.integer(combo)) |>
  mutate(label = factor(LAB[pattern], levels = rev(LAB[pattern])))
p2 <- ggplot(cdf, aes(n, label)) +
  geom_col(fill = PAL["blue"], width = 0.7) +
  geom_text(aes(label = n), hjust = -0.15, size = 2.5) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.22))) +
  labs(x = "compounds", y = NULL, title = "Which gaps occur together")
print(p1 | p2)
print(as.data.frame(cdf[c("pattern", "n")]))



<p class="caption"><b>Figure 1.</b> Left: the missingness map, rows sorted by number of measurements. The <code>pActivity_sd</code> column is dark for the upper block — the single-measurement compounds — and light below, a perfectly structured pattern. Right: the gap combinations. The dominant pattern is "name, phase and sd all missing" (3,611 compounds: unnamed molecules measured once), followed by "name and phase missing but sd present" (372) and "sd only" (179). Only 70 compounds have no gap at all. Gaps that co-occur in a block like this are never random; each combination corresponds to a recognisable kind of compound, which is exactly the diagnosis Unit 4 called <em>structured</em> rather than completely random missingness.</p>



In [ ]:
bounds <- list(pActivity = c(2, 12), MolWt = c(50, 2000), MolLogP = c(-10, 12),
               TPSA = c(0, 500), HBD = c(0, 30), HBA = c(0, 40), RotB = c(0, 60),
               QED = c(0, 1), FractionCSP3 = c(0, 1))
viol <- vapply(names(bounds), function(c_) {
  b <- bounds[[c_]]; sum(cmpd[[c_]] < b[1] | cmpd[[c_]] > b[2]) }, integer(1))
cat("values outside physically possible bounds:",
    if (any(viol > 0)) paste(names(viol)[viol > 0], viol[viol > 0]) else "none", "\n\n")
out <- lapply(c("pActivity", "MolWt", "MolLogP", "TPSA", "RotB", "QED"), function(c_) {
  v <- cmpd[[c_]]; z <- (v - mean(v)) / sd(v)
  q <- unname(quantile(v, c(.25, .75))); f <- 1.5 * (q[2] - q[1])
  data.frame(column = c_, n_z_gt_3 = sum(abs(z) > 3),
             n_tukey = sum(v < q[1] - f | v > q[2] + f),
             min = min(v), max = max(v))
}) |> do.call(what = rbind)
print(out, row.names = FALSE, digits = 4)
h <- cmpd[which.max(cmpd$MolWt), ]
cat(sprintf("\nheaviest compound: %.0f Da, %d heavy atoms, pActivity %.2f\n",
            h$MolWt, h$HeavyAtoms, h$pActivity))



<p class="caption"><b>Table (Section 1, outliers).</b> No value violates a physical bound — a reassuring but necessary check, since a negative TPSA or a QED above 1 would mean a corrupt file rather than an unusual molecule. The two outlier rules disagree systematically: for <code>MolWt</code> the z-score rule (|z| &gt; 3) flags 20 points and Tukey's 1.5·IQR fence 36, and for <code>MolLogP</code> 14 against 66, because the standard deviation is itself inflated by the outliers it is meant to detect. <code>pActivity</code> is the reverse — 4 by z-score and none by the fence, since its interquartile range is wide. Neither rule is a licence to delete. The heaviest compound here is 990 Da with 69 heavy atoms, twice the size of a typical drug but a real peptide-like molecule, not an error; its pActivity of 4.52 is the "&gt; 30 µM" censoring limit.</p>

## 2. The grammar of graphics

Before plotting anything, it is worth having a vocabulary. Wilkinson's *grammar of graphics*, as refined by Wickham,<sup class="c"><a href="#r1">[1]</a></sup> says that any statistical graphic is built from independent layers:

- **Data** — the tidy table.
- **Aesthetic mapping** — which column drives which visual channel: position *x*, position *y*, colour, size, shape, transparency.
- **Geometry** — the mark that is drawn: point, line, bar, box, area.
- **Statistical transformation** — what is computed before drawing: identity, a count per bin, a density, a smoother.
- **Scales** — how data values become channel values: linear or logarithmic axis, a colour map, a size range.
- **Facets** — the rule that splits the data into small multiples.
- **Coordinates and theme** — the coordinate system and the non-data ink.

The value of the grammar is that it makes figure design a series of explicit choices rather than a search through a menu of chart types. "Histogram" is not a chart type; it is *geometry = bar, statistic = count per bin, x = the variable, y = the count*. Change the statistic to "density" and you have a density plot; change the geometry to "step" and you have a frequency polygon; add a facet and you have small multiples of all three.

R is where this is more than a metaphor: `ggplot2` *is* an implementation of the grammar, so every layer above appears literally in the code. `aes()` is the aesthetic mapping, `geom_*()` the geometry, `stat_*()` the statistical transformation, `scale_*()` the scales, `facet_*()` the facets, `coord_*()` the coordinate system and `theme()` the non-data ink. A reader who can name the layers can read any ggplot call, and — more usefully — can write one without searching for a chart type. The practical consequence is that changing a figure means changing one layer, not starting again; the distinction between a figure made to think with and a figure made to publish stays a matter of the theme and the labels.<sup class="c"><a href="#r28">[28]</a></sup> Both require the table to be tidy in the sense of Unit 4.<sup class="c"><a href="#r7">[7]</a></sup>



In [ ]:
d <- cmpd |> filter(!censored) |>
  mutate(size_class = cut(MolWt, c(0, 350, 450, 2000),
                          labels = c("small", "medium", "large")))
base <- ggplot(d, aes(MolLogP, pActivity))
g1 <- base + geom_point(size = 0.4, alpha = 0.15, colour = PAL["blue"]) +
  labs(title = "geom = point\nstat = identity")
g2 <- base + geom_bin_2d(bins = 28) + scale_fill_viridis_c(guide = "none") +
  labs(title = "geom = tile\nstat = count in bin")
g3 <- ggplot(d, aes(MolLogP, colour = size_class)) + stat_ecdf(linewidth = 0.6) +
  scale_colour_manual(values = unname(PAL[1:3]), name = NULL) +
  labs(y = "cumulative fraction", title = "geom = step\nstat = ECDF, colour = MolWt class") +
  theme(legend.position = "bottom")
g4 <- base + geom_point(size = 0.4, alpha = 0.12, colour = PAL["green"]) +
  geom_smooth(method = "lm", formula = y ~ x, se = FALSE,
              colour = PAL["orange"], linewidth = 0.7) +
  labs(title = "geom = point + line\nstat = linear smoother")
print(g1 | g2 | g3 | g4)
fit <- lm(pActivity ~ MolLogP, data = d)
cat(sprintf("linear fit of pActivity on MolLogP: slope %.3f, r = %.3f, p = %.3g\n",
            coef(fit)[2], cor(d$MolLogP, d$pActivity), summary(fit)$coefficients[2, 4]))



<p class="caption"><b>Figure 2.</b> One dataset (the 2,764 compounds with exact measurements), four figures differing only in the grammar layers named in each title. The scatter shows every point but hides density; the binned version shows density but hides individual points; the ECDF changes both the statistic and the aesthetic mapping (colour now encodes a molecular-weight class) and answers a different question — how the distributions of greasiness compare between size classes; the fourth adds a linear smoother. The fitted slope is −0.025 pActivity units per logP unit with r = −0.025 and p = 0.18 — no relation at all. Greasiness does not predict potency here, and the regression line exists only to make that visible. The <code>hexbin</code> package is not available in this environment, so the second panel uses rectangular bins (<code>geom_bin_2d</code>) where the Python edition uses hexagons; the statistic is the same.</p>

The faceting layer deserves its own demonstration, because it is what turns a question about subgroups into one line of code.



In [ ]:
plot_df <- cmpd |>
  mutate(measured = ifelse(censored, "censored only", "has exact value"),
         replicated = ifelse(n_measurements > 1, "replicated", "single measurement"))
print(ggplot(plot_df, aes(pActivity, fill = measured)) +
  geom_histogram(binwidth = 0.25, boundary = 3, colour = NA) +
  facet_wrap(~replicated, scales = "free_y") +
  scale_fill_manual(values = c("has exact value" = unname(PAL["blue"]),
                               "censored only" = unname(PAL["orange"])), name = NULL) +
  labs(x = "pActivity", y = "compounds",
       title = "One aesthetic, one geometry, one facet: the grammar made concrete") +
  theme(legend.position = "bottom"))
print(as.data.frame(plot_df |> group_by(replicated, measured) |>
  summarise(n = n(), median = median(pActivity), .groups = "drop")))
cat(sprintf("censored fraction: single-measurement %.0f%%, replicated %.0f%%\n",
            100 * mean(plot_df$censored[plot_df$replicated == "single measurement"]),
            100 * mean(plot_df$censored[plot_df$replicated == "replicated"])))



<p class="caption"><b>Figure 3.</b> The same table, split by whether a compound was measured once or more than once. One call declares the aesthetic mapping (<code>x</code>, <code>fill</code>), the statistic (<code>geom_histogram</code>, i.e. count per bin), the facet (<code>facet_wrap</code>) and the scales. The two panels have visibly different compositions — 37% of the 3,788 single-measurement compounds are censored-only against 24% of the 492 replicated ones, because a compound worth measuring twice was usually worth measuring at all. That is the grammar's payoff: the question "does the censoring rate depend on replication?" became one argument. Note <code>scales = "free_y"</code>, used here because the panels differ eightfold in size; it is also the decision that makes the two panels non-comparable in height, which the caption must therefore say.</p>

## 3. Distributions: histogram, density, ECDF — and what each hides

The histogram is the first plot anyone draws and the one with the largest hidden parameter: the **bin width**. Two standard rules exist. Scott's rule<sup class="c"><a href="#r12">[12]</a></sup> chooses \( h = 3.49\,\sigma\,n^{-1/3} \), optimal for data that are roughly normal; the Freedman–Diaconis rule<sup class="c"><a href="#r13">[13]</a></sup> uses \( h = 2\,\mathrm{IQR}\,n^{-1/3} \), which is robust to outliers because the interquartile range is. Neither is right for multimodal data, where the bin width that reveals the modes is a judgement.<sup class="c"><a href="#r14">[14]</a></sup> The honest practice is to look at several and say which one you are showing. R's `hist()` defaults to Sturges' rule, which is known to under-bin large samples; `ggplot2` refuses to choose at all and warns until you set `binwidth` — an irritation that is really a feature.



In [ ]:
y <- d$pActivity; n <- length(y)
h_scott <- 3.49 * sd(y) * n^(-1/3)
h_fd    <- 2 * IQR(y) * n^(-1/3)
cat(sprintf("n = %d; Scott bin width = %.3f; Freedman-Diaconis = %.3f log units\n",
            n, h_scott, h_fd))
panels <- list(c(1.0, 0), c(h_scott, 1), c(h_fd, 2), c(0.05, 3))
titles <- c("h = 1.0 (too wide)", sprintf("Scott, h = %.2f", h_scott),
            sprintf("Freedman-Diaconis, h = %.2f", h_fd), "h = 0.05 (too narrow)")
plots <- Map(function(p, ttl) {
  ggplot(d, aes(pActivity)) +
    geom_histogram(binwidth = p[1], boundary = 3, fill = PAL["blue"],
                   colour = "white", linewidth = 0.1) +
    labs(x = "pActivity", y = "compounds", title = ttl)
}, panels, titles)
print(wrap_plots(plots, nrow = 1))
cat("tallest 0.05-wide bin:",
    max(table(cut(y, seq(3.3, 10.0, by = 0.05)))), "compounds\n")



<p class="caption"><b>Figure 4.</b> The potency distribution of the same 2,764 compounds at four bin widths. At h = 1.0 the distribution looks like a smooth single hump. At the Scott and Freedman–Diaconis widths (0.28 and 0.22 log units) a broad, slightly right-skewed unimodal shape is visible (Fisher skewness +0.06; mean 6.51 above median 6.44). At h = 0.05 the histogram dissolves into spikes — but those spikes are not noise: the tallest 0.05-wide window holds 202 compounds, far more than a smooth distribution would put in a window that narrow, and Section 11 traces the cause to a set of 141 molecules reported with one identical potency. A bin width too small is usually useless; here it was the first clue to an anomaly.</p>

Three displays answer the "what is the distribution?" question, and they fail in different ways. The **histogram** depends on bin width *and* bin origin. The **kernel density estimate** replaces bins with a smooth kernel and depends on a bandwidth; it also smooths across hard boundaries, inventing density where none can exist. The **empirical cumulative distribution function** has no free parameter at all: it plots, for every observed value, the fraction of observations at or below it. The ECDF is the only one of the three that shows the data exactly, and the only one most readers cannot read fluently — which is the trade.



In [ ]:
dens <- lapply(c(0.3, 1.0, 3.0), function(adj) {
  k <- density(y, adjust = adj, n = 400)
  tibble::tibble(x = k$x, dens = k$y, bw = sprintf("adjust = %.1f", adj))
}) |> bind_rows()
p1 <- ggplot(d, aes(pActivity)) +
  geom_histogram(aes(y = after_stat(density)), binwidth = h_fd, boundary = 3,
                 fill = "grey85", colour = "white", linewidth = 0.1) +
  geom_line(data = dens, aes(x, dens, colour = bw), linewidth = 0.55) +
  geom_vline(xintercept = min(y), linetype = "dotted", colour = "grey30") +
  scale_colour_manual(values = unname(PAL[c(2, 1, 3)]), name = NULL) +
  labs(x = "pActivity", y = "density", title = "Histogram and three bandwidths") +
  theme(legend.position = "bottom", legend.text = element_text(size = 6))
q <- quantile(y, c(.25, .5, .75))
p2 <- ggplot(d, aes(pActivity)) + stat_ecdf(colour = PAL["blue"], linewidth = 0.6) +
  geom_hline(yintercept = c(.25, .5, .75), colour = "grey80", linewidth = 0.25) +
  annotate("text", x = q, y = c(.25, .5, .75), label = sprintf("%.2f", q),
           hjust = -0.2, vjust = 1.5, size = 2.4) +
  labs(x = "pActivity", y = "cumulative fraction", title = "ECDF: no free parameter")
p3 <- ggplot(cmpd, aes(pActivity, colour = ifelse(censored, "censored-only", "exact"))) +
  stat_ecdf(linewidth = 0.6) +
  scale_colour_manual(values = c(exact = unname(PAL["blue"]),
                                 `censored-only` = unname(PAL["orange"])), name = NULL) +
  labs(x = "pActivity", y = "cumulative fraction", title = "Two populations on one ECDF") +
  theme(legend.position = "bottom")
print(p1 | p2 | p3)
cat(sprintf("quartiles of pActivity (exact compounds): %.2f / %.2f / %.2f\n", q[1], q[2], q[3]))
cat(sprintf("censored-only compounds at exactly pActivity 5.00: %.0f%%\n",
            100 * mean(abs(cmpd$pActivity[cmpd$censored] - 5) < 1e-9)))



<p class="caption"><b>Figure 5.</b> Left: the histogram with three kernel bandwidths over it (<code>density(adjust = …)</code>, where 1 is R's rule-of-thumb default). At adjust = 0.3 the curve is lumpy and reproduces every reporting spike; at 3 it is a featureless hump in which the pile-up at 6.26 has vanished entirely; the default sits between the two and still shows the spike. All three extend past the dotted line marking the smallest observed value, assigning density to potencies that were never measured — the boundary artefact that makes a density estimate the wrong choice for bounded data unless the boundary is handled explicitly. Middle: the ECDF of the same values, with the quartiles (5.73, 6.44, 7.30) read directly off the curve; it has no tuning parameter and no artefact. Right: the ECDF makes the two populations immediately comparable — the censored-only compounds rise almost vertically at pActivity 5.00, because 42% of them sit exactly there, the "&gt; 10 µM" screening limit. A histogram of the pooled data would show this as a mysterious spike; the ECDF shows it as what it is, a point mass.</p>

## 4. Comparing groups: box, violin, strip — and how each misleads

Checklist item 5 asks whether the variable of interest differs between groups. In this dataset the natural grouping is the **scaffold family**: compounds sharing a Bemis–Murcko scaffold are chemical relatives, usually from the same paper and the same optimisation campaign.

The box plot<sup class="c"><a href="#r16">[16]</a></sup> summarises each group with five numbers and is unbeatable for comparing many groups at once. It also hides everything that is not those five numbers: the sample size, multimodality, and gaps. The violin plot<sup class="c"><a href="#r15">[15]</a></sup> shows a mirrored density estimate and therefore inherits the bandwidth and boundary problems of Section 3. The strip (or jitter) plot shows every point and fails only when there are too many. Nature Methods and the biomedical literature have been arguing for a decade that bar charts and box plots should be replaced by, or at least combined with, the raw points,<sup class="c"><a href="#r17">[17]</a></sup><sup class="c"><a href="#r18">[18]</a></sup><sup class="c"><a href="#r19">[19]</a></sup><sup class="c"><a href="#r20">[20]</a></sup> and the "raincloud" combination of half-violin, box and points is the current consensus display.<sup class="c"><a href="#r21">[21]</a></sup>



In [ ]:
top6 <- names(sort(table(cmpd$scaffold_smiles), decreasing = TRUE))[1:6]
fam <- d |> filter(scaffold_smiles %in% top6) |>
  mutate(family = paste0("S", match(scaffold_smiles, top6)))
bp <- ggplot(fam, aes(family, pActivity))
q1 <- bp + geom_boxplot(fill = "grey85", outlier.shape = NA, linewidth = 0.3) + labs(title = "box")
q2 <- bp + geom_violin(fill = unname(PAL["green"]), alpha = 0.7, linewidth = 0.25) +
  stat_summary(fun = median, geom = "point", size = 0.8) + labs(title = "violin")
set.seed(0)
q3 <- bp + geom_jitter(width = 0.22, height = 0, size = 0.5, alpha = 0.5,
                       colour = PAL["blue"]) + labs(title = "strip (all points)")
q4 <- bp + geom_boxplot(fill = unname(PAL["orange"]), alpha = 0.6, outlier.shape = NA,
                        linewidth = 0.25, width = 0.5) +
  geom_jitter(width = 0.12, height = 0, size = 0.4, alpha = 0.45, colour = PAL["blue"]) +
  labs(title = "box + points")
print((q1 | q2 | q3 | q4) & labs(x = "scaffold family"))
print(as.data.frame(fam |> group_by(family) |>
  summarise(n = n(), median = median(pActivity), sd = sd(pActivity),
            lo = min(pActivity), hi = max(pActivity), .groups = "drop") |>
  mutate(across(where(is.numeric), \(x) round(x, 2)))))



<p class="caption"><b>Figure 6.</b> Six scaffold families, four displays. The box plot (left) makes the comparison instantly: family S1 (median 7.52) is more than a log unit above S3 (6.23) and almost two and a half above S2 (5.10). What it does not show is that S1 contains 207 compounds and S6 only 16 — boxes of equal width standing for samples that differ thirteenfold in size. The violin adds shape but smooths S2 into a symmetric blob, when in fact all 23 of its compounds lie between 4.65 and 5.10 with a hard ceiling at 5.10, and it does the same to S6, whose 16 compounds end at exactly 6.26. Only the strip and the box-with-points show both: the family sizes, and the fact that S2 and S6 are walls of points against a limit rather than distributions. Always draw the points when the group has fewer than a few hundred members.</p>

The strongest demonstration is a constructed one: two samples with *identical* box plots and completely different distributions.



In [ ]:
set.seed(1)
m <- 400
uni <- rnorm(m, 6.4, 0.95)
qq <- unname(quantile(uni, c(.25, .5, .75)))     # match the quartiles by construction
bim <- c(rnorm(m / 2, qq[1], 0.30), rnorm(m / 2, qq[3], 0.30))
print(data.frame(stat = c("min", "Q1", "median", "Q3", "max"),
                 unimodal = round(quantile(uni, c(0, .25, .5, .75, 1)), 2),
                 bimodal  = round(quantile(bim, c(0, .25, .5, .75, 1)), 2)),
      row.names = FALSE)
two <- tibble::tibble(value = c(uni, bim),
                      sample = rep(c("A (unimodal)", "B (bimodal)"), each = m))
r1 <- ggplot(two, aes(sample, value)) +
  geom_boxplot(fill = "grey85", outlier.shape = NA, linewidth = 0.3) +
  labs(x = NULL, title = "Box plots: nearly the same")
r2 <- ggplot(two, aes(value, colour = sample)) +
  geom_freqpoly(binwidth = 0.2, linewidth = 0.6) +
  scale_colour_manual(values = unname(PAL[c(1, 2)]), name = NULL) +
  labs(y = "count", title = "Histograms: not the same at all") +
  theme(legend.position = "bottom")
print(r1 | r2)
cat(sprintf("excess kurtosis: A = %.2f, B = %.2f (a strongly negative value flags bimodality)\n",
            kurtosis(uni, type = 2), kurtosis(bim, type = 2)))



<p class="caption"><b>Figure 7.</b> Two samples of 400 values with almost identical quartiles and therefore almost identical box plots, but entirely different shapes: A is a single hump, B is two separated clusters. No five-number summary can tell them apart, and no statistical test applied to the medians would notice. The excess kurtosis does flag the difference — −0.01 for A against −1.22 for B, the strongly negative value being the signature of two separated clusters — but only because we thought to compute it. The general lesson: a box plot answers "where is the middle and how wide is the spread", and nothing else.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Replace the box plots in Figure 6 by violins with <code>geom_violin(adjust = 0.2)</code> and again with <code>adjust = 1.5</code>. Which families change appearance most, and why are they the small ones? Then plot family S2 alone with <code>stat_ecdf()</code> and state in one sentence what the violin was hiding.</p></div>



In [ ]:
bimodality_coefficient <- function(v) {
  nn <- length(v)
  g1 <- skewness(v, type = 2); g2 <- kurtosis(v, type = 2)
  (g1^2 + 1) / (g2 + 3 * (nn - 1)^2 / ((nn - 2) * (nn - 3)))
}
for (p in list(list("exact compounds only", cmpd$pActivity[!cmpd$censored]),
               list("all compounds (pooled)", cmpd$pActivity),
               list("constructed bimodal sample B", bim))) {
  b <- bimodality_coefficient(p[[2]])
  cat(sprintf("%-30s n=%-5d b = %.3f  %s\n", p[[1]], length(p[[2]]), b,
              if (b > 5/9) "-> bimodal" else ""))
}
cat(sprintf("\ncompounds at exactly pActivity 5.00: %d (%.1f%% of the pooled set)\n",
            sum(abs(cmpd$pActivity - 5) < 1e-9), 100 * mean(abs(cmpd$pActivity - 5) < 1e-9)))



Read the three numbers carefully, because they make two points at once. Sarle's coefficient orders the samples correctly — 0.391 for the exact compounds, 0.500 for the pooled set, 0.556 for the constructed mixture of Figure 7 — so pooling the censored compounds does move the distribution towards bimodality. But the sample we *built* to be bimodal, and can see is bimodal in Figure 7, clears the usual \( 5/9 \approx 0.5556 \) reference value by three units in the fourth decimal, and the pooled real data do not reach it at all. (The Python edition, drawing its mixture from a different random stream, got 0.528 and did not cross the line; a diagnostic whose verdict flips with the random seed is a diagnostic to distrust.) A one-number test for shape is weak, which is why Hartigan's dip test exists as the formal tool<sup class="c"><a href="#r22">[22]</a></sup> (the `diptest` package implements it; it is not installed here, so we report the coefficient and the picture instead) and why neither is a substitute for the plot. The real diagnosis is mechanistic: the second mode of the pooled distribution is the 658 compounds sitting at exactly pActivity 5.00, the "&gt; 10 µM" screening limit, so the apparent bimodality of "the ROCK1 potency distribution" is a property of the experiments, not of the chemistry.

## 5. Relationships and the overplotting problem

Checklist item 4 is the pairwise structure. A scatter plot of 4,280 points is already past the density at which a scatter plot works: the ink saturates, and the eye reads the *outline* of the cloud rather than its density. Three standard repairs exist — transparency, binning, and subsampling — and each trades something different.



In [ ]:
bb <- ggplot(cmpd, aes(MolWt, MolLogP)) + coord_cartesian(xlim = c(100, 800))
o1 <- bb + geom_point(size = 1.1, colour = PAL["blue"]) +
  labs(title = "opaque points\n(the cloud has no inside)")
o2 <- bb + geom_point(size = 0.5, alpha = 0.08, colour = PAL["blue"]) +
  labs(title = "alpha = 0.08\n(density visible, outliers faint)")
o3 <- bb + geom_bin_2d(bins = 35) +
  scale_fill_viridis_c(name = "compounds") +
  labs(title = "2-D bins\n(density quantified)") +
  theme(legend.position = "bottom", legend.key.width = unit(0.5, "cm"))
set.seed(0)
o4 <- ggplot(slice_sample(cmpd, n = 400), aes(MolWt, MolLogP)) +
  coord_cartesian(xlim = c(100, 800)) +
  geom_point(size = 0.7, alpha = 0.7, colour = PAL["orange"]) +
  labs(title = "random 400\n(honest but throws data away)")
print((o1 | o2 | o3 | o4) & labs(x = "MolWt (Da)", y = "MolLogP"))
bins <- ggplot_build(o3)$data[[1]]
cat("points:", nrow(cmpd), "| densest 2-D bin holds", max(bins$count), "compounds\n")
cat(sprintf("Pearson r(MolWt, MolLogP) = %.3f | Spearman = %.3f\n",
            cor(cmpd$MolWt, cmpd$MolLogP), cor(cmpd$MolWt, cmpd$MolLogP, method = "spearman")))



<p class="caption"><b>Figure 8.</b> Four views of molecular weight against lipophilicity for all 4,280 compounds. The opaque scatter is a silhouette: it shows the support of the distribution and nothing about where the mass is. Transparency recovers the density but makes isolated compounds — the interesting ones — nearly invisible. The 2-D binning quantifies the density, with the busiest cell holding 94 compounds, but it discards the individual points entirely and introduces its own bin-size parameter. The 400-point subsample is readable and honest if the caption says so. In practice: transparency plus a separate marked layer for the points you want seen, or binning plus a note of the maximum cell count. The correlation between weight and lipophilicity here is 0.608 (Pearson, 0.617 Spearman) — a real but loose relation of the kind a scatter plot communicates far better than a single number.</p>

<div class="box bridge"><h4>Domain context — what a medicinal chemist looks at first</h4>
<p>Faced with a new activity dataset, a medicinal chemist looks for three things. First, the <strong>potency distribution</strong>: how many compounds are genuinely potent (pActivity above about 7, i.e. below 100 nM) and how many are at the screening floor. Second, the <strong>property distributions</strong> against the ranges associated with orally absorbed drugs — Lipinski's rule of five, which observes that most oral drugs have molecular weight below 500, logP below 5, at most 5 hydrogen-bond donors and at most 10 acceptors.<sup class="c"><a href="#r31">[31]</a></sup> Third, <strong>activity cliffs</strong>: pairs of nearly identical molecules with very different potencies, which are both the most informative compounds in a series and the ones that break similarity-based models.<sup class="c"><a href="#r33">[33]</a></sup><sup class="c"><a href="#r34">[34]</a></sup></p></div>

## 6. Correlation structure

Checklist item 4 again, now for every pair at once. Two cautions before drawing a correlation matrix. **Pearson's** coefficient measures linear association and is sensitive to outliers; **Spearman's** measures monotone association by correlating ranks and is not. Report which you used. And remember that a correlation matrix shows only pairwise relations: variables can be strongly dependent with zero correlation.



In [ ]:
feat <- c("pActivity", "MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotB",
          "AromaticRings", "RingCount", "HeavyAtoms", "FractionCSP3", "QED")
P <- cor(d[feat]); S <- cor(d[feat], method = "spearman")
op <- par(mfrow = c(1, 2), mar = c(0.5, 0.5, 2, 0.5))
for (k in 1:2) {
  M <- if (k == 1) P else S
  corrplot(M, method = "color", type = "full", tl.col = "black", tl.cex = 0.55,
           tl.srt = 90, cl.cex = 0.5, number.cex = 0.42, addCoef.col = "grey20",
           col = colorRampPalette(c("#b2182b", "white", "#2166ac"))(100),
           mar = c(0, 0, 1.5, 0), title = if (k == 1) "Pearson" else "Spearman")
}
par(op)
pp <- P; diag(pp) <- 0
idx <- which(upper.tri(pp), arr.ind = TRUE)
ordp <- order(abs(pp[upper.tri(pp)]), decreasing = TRUE)[1:5]
cat("strongest descriptor pairs (Pearson):\n")
for (k in ordp) cat(sprintf("  %-14s ~ %-14s %.3f\n", feat[idx[k, 1]], feat[idx[k, 2]],
                            pp[idx[k, 1], idx[k, 2]]))
cat("\ncorrelation of each descriptor with pActivity:\n")
tr <- data.frame(pearson = P["pActivity", ], spearman = S["pActivity", ])
print(round(tr[rownames(tr) != "pActivity", ][order(tr$pearson[rownames(tr) != "pActivity"]), ], 3))



<p class="caption"><b>Figure 9.</b> Pearson and Spearman correlation matrices of eleven descriptors plus the target, drawn with <code>corrplot</code>. The two are visually almost identical here, which is itself the finding: the relations are monotone and not driven by outliers. The size family dominates — <code>MolWt</code> and <code>HeavyAtoms</code> correlate at 0.979, so they carry one piece of information between them, and <code>QED</code> is strongly anti-correlated with both (−0.803 and −0.807) because its definition penalises large molecules. The target row is almost white: the strongest correlation of any descriptor with potency is <code>RotB</code> at 0.241, next <code>FractionCSP3</code> at 0.114, and everything else is below 0.11 in absolute value. Simple descriptors do not predict ROCK1 potency — which is precisely why Units 11–17 use structural fingerprints instead.</p>

A **clustered heatmap** reorders rows and columns by hierarchical clustering so that correlated blocks become contiguous. On twelve variables the gain is modest; on a hundred it is the difference between a readable figure and a grey square.



In [ ]:
pheatmap::pheatmap(P, color = colorRampPalette(c("#b2182b", "white", "#2166ac"))(100),
                   breaks = seq(-1, 1, length.out = 101),
                   clustering_distance_rows = as.dist(1 - P),
                   clustering_distance_cols = as.dist(1 - P),
                   clustering_method = "complete", fontsize = 7,
                   treeheight_row = 24, treeheight_col = 24, border_color = NA,
                   main = "Descriptor correlations, hierarchically clustered")
hc <- hclust(as.dist(1 - P), method = "complete")
cat("column order after clustering:", paste(feat[hc$order], collapse = ", "), "\n")



<p class="caption"><b>Figure 10.</b> The same correlation matrix as a clustered heatmap. The reordering brings the recognisable families together: <code>MolWt</code> beside <code>HeavyAtoms</code>, the polarity group <code>TPSA</code>/<code>HBD</code>/<code>HBA</code>, and <code>MolLogP</code> beside <code>AromaticRings</code> and <code>RingCount</code>. <code>pActivity</code> is placed next to <code>FractionCSP3</code> — not because it is related to it, but because a variable correlated with nothing has no natural neighbour and the algorithm must still place it somewhere. That is the main caution about dendrograms: every leaf gets a position whether or not it has a reason to be there, and the branch order depends on the linkage and distance chosen. The Python edition, using seaborn's defaults (Euclidean distance on the correlation rows, average linkage), produced a different leaf order from the same matrix — the same caution, demonstrated twice.</p>

## 7. Small multiples and scales

Faceting — Tufte's *small multiples* — answers "does this relation hold in every subgroup?" by repeating one small plot across the levels of a variable, with shared axes so the panels are comparable. Shared axes are the whole point; panels with independent axes are four unrelated figures in a grid, which is why `facet_wrap` shares them by default and `scales = "free"` has to be asked for.

The other decision is the **scale**. Potency data are log-normal-ish over six orders of magnitude, which is why the field works in pActivity at all; plotting raw nanomolar concentrations on a linear axis compresses almost every compound into the leftmost pixel column.



In [ ]:
cmpd <- cmpd |> mutate(potency_class = cut(pActivity, c(0, 5.5, 6.5, 7.5, 12),
                       labels = c("weak (<5.5)", "moderate", "good", "potent (>7.5)")))
lab_n <- cmpd |> count(potency_class) |> mutate(lab = paste0(potency_class, "  (n=", n, ")"))
top <- ggplot(left_join(cmpd, select(lab_n, potency_class, lab), by = "potency_class",
                        relationship = "many-to-one"),
              aes(MolWt, TPSA)) +
  geom_point(size = 0.4, alpha = 0.35, colour = PAL["blue"]) +
  geom_vline(xintercept = 500, colour = PAL["orange"], linetype = "dashed", linewidth = 0.3) +
  facet_wrap(~lab, nrow = 1) + coord_cartesian(xlim = c(100, 800)) +
  labs(x = "MolWt (Da)", y = "TPSA", title = "Shared axes make the panels comparable")
ic50 <- tibble::tibble(nM = 10^(9 - d$pActivity), pAct = d$pActivity)
b1 <- ggplot(ic50, aes(nM)) + geom_histogram(bins = 60, fill = PAL["orange"]) +
  labs(x = "IC50 (nM)", y = "compounds", title = "linear concentration axis")
b2 <- ggplot(ic50, aes(nM)) + geom_histogram(bins = 60, fill = PAL["green"]) +
  scale_x_log10() + labs(x = "IC50 (nM), log scale", y = NULL, title = "logarithmic axis")
b3 <- ggplot(ic50, aes(pAct)) + geom_histogram(bins = 40, fill = PAL["blue"]) +
  labs(x = "pActivity", y = NULL, title = "pActivity (= the same log scale)")
print(top / (b1 | b2 | b3))
cat(sprintf("fraction of compounds below one-thirtieth of the largest IC50: %.1f%%\n",
            100 * mean(ic50$nM < max(ic50$nM) / 30)))
print(as.data.frame(cmpd |> group_by(potency_class) |>
  summarise(n = n(), median_MolWt = round(median(MolWt), 1), .groups = "drop")))



<p class="caption"><b>Figure 11.</b> Top row: four small multiples of polar surface area against molecular weight, one per potency class, with shared axes and Lipinski's 500-dalton line marked. The clouds are nearly indistinguishable — potent and weak compounds occupy the same property space, consistent with the near-zero correlations of Figure 9; the only systematic movement is a mild decline in median molecular weight, from 405.4 Da in the weak class (n = 1,667) to 370.9 Da in the potent one (n = 577). Bottom row: the same 2,764 potencies as raw nanomolar concentrations on a linear axis (useless: 91.2% of compounds fall below one-thirtieth of the largest value, so the axis is dominated by a handful of millimolar outliers), on a logarithmic axis, and as pActivity. The last two are the same picture reflected, which is why the field reports the logarithm.</p>

## 8. Same statistics, different graphs

Anscombe published four small datasets in 1973 with identical means, variances, correlation and regression line, and four completely different scatter plots.<sup class="c"><a href="#r9">[9]</a></sup> The point was not a curiosity; it was an argument that numerical summaries are insufficient and that looking is a required step. R ships the quartet in its `datasets` package, which saves us typing it — and means we are reading exactly the numbers Anscombe published.

<h4>Worked example by hand</h4>

Take Anscombe's set I: *x* = 10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5. Their sum is 99, so the mean is 99/11 = **9.00**. The deviations from the mean are 1, −1, 4, 0, 2, 5, −3, −5, 3, −2, −4; their squares sum to 1+1+16+0+4+25+9+25+9+4+16 = **110**, so the sample variance is 110/10 = **11.00** and the standard deviation \( \sqrt{11} = 3.3166 \). Sets I–III share the *x* column entirely, so these three numbers are identical by construction; set IV has a different *x* column chosen to give the same mean and variance. The chunk below confirms all of this and draws the four pictures.



In [ ]:
ans <- lapply(1:4, function(i) {
  x <- anscombe[[paste0("x", i)]]; yv <- anscombe[[paste0("y", i)]]
  fit <- lm(yv ~ x)
  data.frame(set = c("I", "II", "III", "IV")[i], mean_x = mean(x), var_x = var(x),
             mean_y = mean(yv), var_y = var(yv), corr = cor(x, yv),
             slope = coef(fit)[2], intercept = coef(fit)[1])
}) |> do.call(what = rbind)
print(ans, row.names = FALSE, digits = 3)
cat(sprintf("\nhand check of set I: mean_x = 99/11 = %.2f, var_x = 110/10 = %.2f, sd = %.4f\n",
            sum(anscombe$x1) / 11, sum((anscombe$x1 - 9)^2) / 10, sqrt(11)))



The table confirms the hand calculation and the construction: every set has mean *x* = 9.00, variance of *x* = 11.00, mean *y* = 7.50, variance of *y* ≈ 4.13, correlation 0.816 and the regression line *y* = 3.00 + 0.50*x*, all agreeing to two decimals. Now the pictures.



In [ ]:
alng <- lapply(1:4, function(i) data.frame(set = paste("set", c("I", "II", "III", "IV")[i]),
                                           x = anscombe[[paste0("x", i)]],
                                           y = anscombe[[paste0("y", i)]])) |>
  do.call(what = rbind)
print(ggplot(alng, aes(x, y)) +
  geom_abline(intercept = 3, slope = 0.5, colour = PAL["orange"], linewidth = 0.5) +
  geom_point(size = 1.3, colour = PAL["blue"]) +
  facet_wrap(~set, nrow = 1) +
  coord_cartesian(xlim = c(2, 20), ylim = c(2, 14)) +
  labs(title = "Anscombe's quartet: one regression line, four datasets"))



<p class="caption"><b>Figure 12.</b> Anscombe's quartet. The orange line is literally the same line, <em>y</em> = 3.00 + 0.50<em>x</em>, drawn once and repeated in all four panels — it is each set's least-squares fit. Set I is a plausible linear relation with scatter; set II is a perfect parabola; set III is a perfect line plus one outlier that drags the fit; set IV is a vertical stack plus a single leverage point that determines the slope entirely. Any of the four would be reported identically by a table of statistics.</p>

Matejka and Fitzmaurice turned Anscombe's hand construction into an algorithm: start from a cloud with the desired statistics, repeatedly nudge one point towards a target shape, and accept the nudge only if the summary statistics still round to the same two decimals.<sup class="c"><a href="#r10">[10]</a></sup> Simulated annealing lets the search escape local traps. Their famous output is a dinosaur; ours, generated here in a few seconds of plain R, is an X and a ring.



In [ ]:
summary5 <- function(a) c(mean(a[, 1]), mean(a[, 2]), sd(a[, 1]), sd(a[, 2]), cor(a[, 1], a[, 2]))
unchanged <- function(a, b, dec = 2) all(round(a, dec) == round(b, dec))
nearest <- function(target, p) min(sqrt((target[, 1] - p[1])^2 + (target[, 2] - p[2])^2))

morph <- function(start, target, iters = 200000, t0 = 0.4, t1 = 0.01, seed = 3, step = 0.8) {
  set.seed(seed); pts <- start; frozen <- summary5(pts); n <- nrow(pts)
  for (i in seq_len(iters)) {
    temp <- t0 + (t1 - t0) * i / iters
    j <- sample.int(n, 1); old <- pts[j, ]
    pts[j, ] <- old + rnorm(2, 0, step)
    if (nearest(target, pts[j, ]) > nearest(target, old) && runif(1) > exp(-1 / temp)) {
      pts[j, ] <- old; next
    }
    if (!unchanged(summary5(pts), frozen)) pts[j, ] <- old
  }
  pts
}
set.seed(0); cloud <- cbind(rnorm(200, 50, 14), rnorm(200, 50, 14))
set.seed(2); u <- runif(200, -1, 1); sgn <- ifelse(runif(200) < .5, 1, -1)
shape_x <- cbind(50 + 24 * u, 50 + 24 * u * sgn)
set.seed(1); ang <- runif(200, 0, 2 * pi)
shape_o <- cbind(50 + 20 * cos(ang), 50 + 20 * sin(ang))
out_x <- morph(cloud, shape_x); out_o <- morph(cloud, shape_o, seed = 4)
tab <- rbind(`random cloud` = summary5(cloud), `X shape` = summary5(out_x),
             ring = summary5(out_o))
colnames(tab) <- c("mean x", "mean y", "sd x", "sd y", "r")
print(round(tab, 2))
pts_df <- rbind(data.frame(cloud, shape = "random cloud"),
                data.frame(out_x, shape = "X"), data.frame(out_o, shape = "ring"))
names(pts_df)[1:2] <- c("x", "y")
pts_df$shape <- factor(pts_df$shape, levels = c("random cloud", "X", "ring"))
print(ggplot(pts_df, aes(x, y)) + geom_point(size = 0.8, colour = PAL["blue"]) +
  facet_wrap(~shape, nrow = 1) +
  labs(title = "Same five statistics to two decimals, three different datasets"))



<p class="caption"><b>Figure 13.</b> Three datasets of 200 points with the same five summary statistics to two decimals — mean x 49.84, mean y 50.49, sd x 12.93, sd y 13.98, correlation 0.02 — and three completely different structures. The table above the figure is the proof; the figure is the point. A referee given only the table could not distinguish a featureless cloud from a ring with a hole in the middle. Run the EDA.</p>

## 9. Colour

Colour is the most-abused channel in scientific figures, and the three failures are well documented.<sup class="c"><a href="#r23">[23]</a></sup> **Non-uniform colour maps** such as the old `jet` create visual boundaries where the data are smooth and hide boundaries where the data jump, because the perceived lightness does not increase monotonically.<sup class="c"><a href="#r24">[24]</a></sup> **Non-accessible palettes** are unreadable to the ~4% of the population — about 8% of men of northern-European descent — with a colour-vision deficiency.<sup class="c"><a href="#r27">[27]</a></sup> **Colour used for nothing** adds ink without adding information.

The rules are short. For a continuous quantity use a perceptually uniform sequential map: `viridis`, `cividis`, `magma`. For a quantity with a meaningful zero use a diverging map centred on it. For categories use a qualitative palette designed for accessibility — the Okabe–Ito eight-colour set is the standard, and base R ships it as `palette.colors(palette = "Okabe-Ito")` — and never more than about seven categories. And always encode the information twice: colour plus marker shape, or colour plus a direct label.<sup class="c"><a href="#r26">[26]</a></sup>

The chunk below simulates how a figure looks to a reader with deuteranopia, using the physiologically derived transformation of Machado, Oliveira and Fernandes.<sup class="c"><a href="#r25">[25]</a></sup>



In [ ]:
M_DEUTAN <- matrix(c(0.367322, 0.860646, -0.227968,
                     0.280085, 0.672501,  0.047413,
                    -0.011820, 0.042940,  0.968881), nrow = 3, byrow = TRUE)
simulate_cvd <- function(hex) {
  rgbv <- t(col2rgb(hex)) / 255
  lin <- ifelse(rgbv <= 0.04045, rgbv / 12.92, ((rgbv + 0.055) / 1.055)^2.4)
  out <- pmin(pmax(lin %*% t(M_DEUTAN), 0), 1)
  srgb <- ifelse(out <= 0.0031308, out * 12.92, 1.055 * out^(1 / 2.4) - 0.055)
  grDevices::rgb(srgb[, 1], srgb[, 2], srgb[, 3])
}
luminance <- function(cols) as.numeric((t(col2rgb(cols)) / 255) %*% c(0.2126, 0.7152, 0.0722))
jet <- colorRampPalette(c("#00007F", "blue", "#007FFF", "cyan", "#7FFF7F",
                          "yellow", "#FF7F00", "red", "#7F0000"))(256)
vir <- viridis(256)
rdbu <- colorRampPalette(c("#053061", "#f7f7f7", "#67001f"))(256)
maps <- list(jet = jet, viridis = vir, RdBu = rdbu)
map_df <- lapply(names(maps), function(nm) {
  L <- luminance(maps[[nm]])
  data.frame(map = nm, i = seq_along(L), col = maps[[nm]], L = L,
             ttl = sprintf("%s: lightness monotone = %s", nm, all(diff(L) > -1e-3)))
}) |> do.call(what = rbind)
map_df$map <- factor(map_df$map, levels = names(maps))
map_df$ttl <- factor(map_df$ttl, levels = unique(map_df$ttl))
pstrip <- ggplot(map_df, aes(i, 1, fill = col)) + geom_raster() +
  scale_fill_identity() + facet_wrap(~ttl, nrow = 1) +
  labs(x = NULL, y = NULL) + theme_void(base_size = 8) +
  theme(strip.text = element_text(size = 7, face = "bold"))
plight <- ggplot(map_df, aes(i, L)) + geom_line(linewidth = 0.4) +
  facet_wrap(~map, nrow = 1) + ylim(0, 1) +
  labs(x = NULL, y = "perceived lightness") +
  theme(axis.text.x = element_blank(), strip.text = element_blank())
OKABE_ITO <- unname(palette.colors(8, "Okabe-Ito"))
bad_palette <- c("#ff0000", "#00b000", "#b06000", "#0000ff",
                 "#ff00ff", "#00c0c0", "#808000", "#c00060")
pal_df <- rbind(
  data.frame(pal = "a typical ad-hoc palette", view = "normal", i = 1:8, col = bad_palette),
  data.frame(pal = "a typical ad-hoc palette", view = "deuteranope", i = 1:8,
             col = simulate_cvd(bad_palette)),
  data.frame(pal = "Okabe-Ito", view = "normal", i = 1:8, col = OKABE_ITO),
  data.frame(pal = "Okabe-Ito", view = "deuteranope", i = 1:8, col = simulate_cvd(OKABE_ITO)))
pal_df$pal <- factor(pal_df$pal, levels = c("a typical ad-hoc palette", "Okabe-Ito"))
ppal <- ggplot(pal_df, aes(i, view, fill = col)) + geom_tile() + scale_fill_identity() +
  facet_wrap(~pal, nrow = 1) + labs(x = NULL, y = NULL) +
  theme(axis.text.x = element_blank(), axis.text.y = element_text(size = 6.5))
print(pstrip / plight / ppal + plot_layout(heights = c(1, 1.4, 1.3)))
min_dist <- function(cols) {
  m <- t(col2rgb(cols)) / 255
  min(dist(m))
}
cat(sprintf("minimum pairwise RGB distance between the 8 categories:\n  ad-hoc, normal vision:  %.2f\n  ad-hoc, deuteranope:    %.2f\n  Okabe-Ito, normal:      %.2f\n  Okabe-Ito, deuteranope: %.2f\n",
            min_dist(bad_palette), min_dist(simulate_cvd(bad_palette)),
            min_dist(OKABE_ITO), min_dist(simulate_cvd(OKABE_ITO))))



<p class="caption"><b>Figure 14.</b> Top two rows: three colour maps with their perceived-lightness profile beneath. <code>jet</code> is not monotone in lightness — it rises, falls and rises again — so equal steps in the data are not equal steps in appearance and spurious bands appear; <code>viridis</code> is monotone by construction; the diverging <code>RdBu</code> is deliberately non-monotone because its midpoint is meaningful. Bottom: eight categorical colours as seen normally and as seen by a deuteranope. In the ad-hoc palette the closest pair of categories collapses from an RGB distance of 0.23 to 0.08 — effectively indistinguishable — whereas the Okabe–Ito set keeps its closest pair at 0.20, barely changed from 0.26. Choosing the palette costs nothing; choosing it badly costs a reader.</p>

## 10. The anti-pattern gallery

Four errors appear in almost every seminar. Each is drawn here as it is usually drawn, then as it should be.

1. **The truncated axis.** A bar chart whose *y* axis does not start at zero exaggerates differences without stating that it does — bar length is the encoding, so truncating it lies. (A truncated axis on a *line* chart of a trend is legitimate and often necessary; the sin is specific to length-encoded geometry.)
2. **The dual axis.** Two series with different units on left and right axes. The apparent relation between them is entirely determined by the scaling the author chose, so the figure can be made to show correlation or anti-correlation at will.
3. **The rainbow map** (Section 9).
4. **The pie chart.** Humans compare angles poorly and lengths well,<sup class="c"><a href="#r11">[11]</a></sup> so a pie with more than three slices is harder to read than the bar chart of the same numbers.



In [ ]:
by_class <- cmpd |> count(potency_class)
binned <- cmpd |> mutate(bin = cut(pActivity, seq(3.5, 10, 0.5))) |>
  filter(!is.na(bin)) |> group_by(bin) |>
  summarise(ctr = mean(as.numeric(sub("\\((.+),.*", "\\1", bin)) + 0.25), n = n(),
            med_mw = median(MolWt), .groups = "drop")
assay_counts <- cmpd |> count(n_assays) |> slice_head(n = 5)

a1 <- ggplot(by_class, aes(potency_class, n)) + geom_col(fill = PAL["orange"], width = .7) +
  coord_cartesian(ylim = c(500, max(by_class$n) * 1.02)) +
  labs(x = NULL, y = "compounds", title = "1. truncated axis") +
  theme(axis.text.x = element_text(angle = 35, hjust = 1, size = 5.5))
a2 <- ggplot(by_class, aes(potency_class, n)) + geom_col(fill = PAL["blue"], width = .7) +
  geom_text(aes(label = n), vjust = -0.3, size = 2.2) +
  scale_y_continuous(limits = c(0, max(by_class$n) * 1.12), expand = c(0, 0)) +
  labs(x = NULL, y = "compounds", title = "fixed: zero baseline, labelled") +
  theme(axis.text.x = element_text(angle = 35, hjust = 1, size = 5.5))
b1 <- ggplot(binned, aes(ctr)) +
  geom_line(aes(y = med_mw), colour = PAL["orange"], linewidth = 0.6) +
  geom_line(aes(y = 330 + n / 9), colour = PAL["blue"], linewidth = 0.6) +
  scale_y_continuous(name = "median MolWt", limits = c(330, 460),
                     sec.axis = sec_axis(~ (. - 330) * 9, name = "compounds")) +
  labs(x = "pActivity", title = "2. dual axis") +
  theme(axis.title = element_text(size = 6), axis.text = element_text(size = 5.5))
b2 <- ggplot(binned, aes(ctr, med_mw)) + geom_line(colour = PAL["blue"], linewidth = 0.6) +
  geom_point(size = 0.8) + geom_text(aes(label = paste0("n=", n)), size = 1.8, vjust = -1) +
  labs(x = "pActivity", y = "median MolWt (Da)", title = "fixed: one axis, n as labels") +
  theme(axis.title = element_text(size = 6), axis.text = element_text(size = 5.5))
H <- cmpd |> mutate(wb = cut(MolWt, 24), lb = cut(MolLogP, 24)) |> count(wb, lb) |>
  mutate(w = as.numeric(wb), l = as.numeric(lb))
c1 <- ggplot(H, aes(w, l, fill = n)) + geom_raster() +
  scale_fill_gradientn(colours = jet, guide = "none") +
  labs(x = "MolWt bin", y = "MolLogP bin", title = "3. rainbow map") +
  theme(axis.title = element_text(size = 6), axis.text = element_text(size = 5.5))
c2 <- ggplot(H, aes(w, l, fill = n)) + geom_raster() +
  scale_fill_viridis_c(name = "compounds") +
  labs(x = "MolWt bin", y = "MolLogP bin", title = "fixed: uniform map + scale") +
  theme(axis.title = element_text(size = 6), axis.text = element_text(size = 5.5),
        legend.position = "bottom", legend.key.width = unit(0.35, "cm"),
        legend.text = element_text(size = 5))
d1 <- ggplot(assay_counts, aes("", n, fill = factor(n_assays))) +
  geom_col(width = 1) + coord_polar(theta = "y") +
  scale_fill_manual(values = c(unname(PAL), "#8c8c8c"), name = "assays") +
  labs(x = NULL, y = NULL, title = "4. pie chart") + theme_void(base_size = 7) +
  theme(plot.title = element_text(face = "bold", size = 9), legend.text = element_text(size = 5.5))
d2 <- ggplot(assay_counts, aes(n, factor(n_assays))) + geom_col(fill = PAL["blue"], width = .7) +
  scale_x_log10() +
  labs(x = "compounds (log)", y = "assays per compound", title = "fixed: bars, log scale") +
  theme(axis.title = element_text(size = 6), axis.text = element_text(size = 5.5))
print((a1 | b1 | c1 | d1) / (a2 | b2 | c2 | d2))
cat("potency classes:\n"); print(as.data.frame(by_class))
cat(sprintf("ratio largest/smallest class: %.2f (the truncated chart makes it look like %.1f)\n",
            max(by_class$n) / min(by_class$n),
            (max(by_class$n) - 500) / (min(by_class$n) - 500)))



<p class="caption"><b>Figure 15.</b> Four anti-patterns (top) and their repairs (bottom). (1) Truncating the bar axis at 500 makes the largest potency class look 15.2 times the smallest when the true ratio is 2.9. (2) The dual-axis panel invites the reader to see median molecular weight and compound count as "tracking each other"; the apparent agreement is produced by the two chosen <em>y</em> ranges and disappears when each series is plotted on its own axis with the counts written as labels. (3) The <code>jet</code> version of the same 2-D histogram invents a bright ridge where the data are smooth; <code>viridis</code> with a colour bar shows the single density peak that is actually there. (4) Five slices of a pie against five bars: the bars are ordered, labelled, and on a log scale that accommodates a range the pie cannot show at all — four of the five categories are invisible slivers.</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Take the dual-axis panel and change only the secondary-axis transformation so that the two curves appear to be <em>anti</em>-correlated. You should be able to do it in one line. Then write the caption you would have to put under that figure for it to be honest — and notice that the honest caption makes the figure pointless.</p></div>

## 11. Three real anomalies

Checklist item 6. An anomaly is not a point to delete but a fact about the experiment that produced the data. We found three.



In [ ]:
counts <- sort(table(round(cmpd$pActivity, 3)), decreasing = TRUE)
cat("most frequent pActivity values and the concentration they correspond to:\n")
for (k in 1:6) {
  v <- as.numeric(names(counts)[k])
  cat(sprintf("  pActivity %-7.3f %4d compounds   = %10.1f nM\n", v, counts[k], 10^(9 - v)))
}
raw <- load_data("rock1_chembl_raw.csv")
pile_ids <- unlist(strsplit(cmpd$chembl_ids[abs(cmpd$pActivity - 6.26) < 1e-9], "\\|"))
rows <- raw |> filter(molecule_chembl_id %in% pile_ids, standard_relation == "=")
cat("\nthe", sum(abs(cmpd$pActivity - 6.26) < 1e-9), "compounds at pActivity 6.26:\n")
cat("  distinct standard_value:", head(unique(rows$standard_value), 4),
    "| assays:", n_distinct(rows$assay_chembl_id),
    "| documents:", n_distinct(rows$document_chembl_id), "\n")
ta <- sort(table(rows$assay_chembl_id), decreasing = TRUE)
cat("  largest contributing assay:", names(ta)[1], "with", ta[1], "of them\n")
cat("  assay text:", substr(rows$assay_description[1], 1, 108), "...\n")



**Anomaly 1 — reported-value pile-ups.** Five values account for an extraordinary share of the table. Two are censoring limits (658 compounds at exactly 5.000, i.e. "> 10 µM", and 264 at 4.523, "> 30 µM"); two are round numbers a chemist would write by hand (205 at exactly 7.000 = 100 nM, 161 at 6.000 = 1 µM); and one, 141 compounds at exactly 6.260, is an IC50 of exactly 550 nM. Identical potencies to three decimals for 141 distinct molecules do not occur by chance. Tracing them through `chembl_ids` back to the raw records shows they come from four assays in four documents, 124 of them from a single assay that reported the same representative value for a whole compound series — a common practice in patent tables, where activities are quoted in bands rather than individually. For modelling this matters twice: those compounds carry far less information than their count suggests, and a random train/test split will put near-identical target values on both sides.



In [ ]:
um <- raw |> filter(units == "uM")
nm <- raw |> filter(units == "nM")
cat("raw rows reported in uM:", nrow(um), "| assays:", n_distinct(um$assay_chembl_id),
    "| documents:", n_distinct(um$document_chembl_id), "\n")
cat(sprintf("their median reported value: %.1f uM (= %.0f nM); median of the nM-reporting rows: %.0f nM\n",
            median(um$value, na.rm = TRUE), 1000 * median(um$value, na.rm = TRUE),
            median(nm$value, na.rm = TRUE)))
cat(sprintf("above 1 uM: %.0f%% of the uM-reported rows vs %.0f%% of the nM-reported ones\n",
            100 * mean(1000 * um$value > 1000, na.rm = TRUE),
            100 * mean(nm$value > 1000, na.rm = TRUE)))
ws <- cmpd |> filter(wide_spread)
cat("\nwide-spread compounds (max - min > 1 log unit):", nrow(ws),
    "of", sum(cmpd$n_exact > 1), "with replicates\n")
cat(sprintf("their range: median %.2f, max %.2f log units\n",
            median(ws$pActivity_range), max(ws$pActivity_range)))
print(as.data.frame(ws |> slice_max(pActivity_range, n = 4) |>
  select(pref_name, pActivity, pActivity_min, pActivity_max, n_measurements, n_documents)))

top8 <- tibble::tibble(pAct = factor(names(counts)[1:8], levels = names(counts)[1:8]),
                       n = as.integer(counts[1:8]))
e1 <- ggplot(top8, aes(pAct, n)) + geom_col(fill = PAL["orange"], width = .7) +
  labs(x = "pActivity", y = "compounds", title = "1. value pile-ups") +
  theme(axis.text.x = element_text(angle = 60, hjust = 1, size = 6))
unit_df <- rbind(data.frame(lv = log10(1000 * um$value), unit = "reported in uM"),
                 data.frame(lv = log10(pmax(nm$value, 1e-2)), unit = "reported in nM")) |>
  filter(is.finite(lv))
e2 <- ggplot(unit_df, aes(lv, colour = unit)) + geom_freqpoly(bins = 25, linewidth = 0.6) +
  scale_colour_manual(values = unname(PAL[c("green", "blue")]), name = NULL) +
  labs(x = "log10(value in nM)", y = "measurements",
       title = "2. the uM assays are the weak ones") +
  theme(legend.position = "bottom")
repl <- cmpd |> filter(n_exact > 1)
e3 <- ggplot(repl, aes(pActivity, pActivity_range)) +
  geom_point(size = 0.6, alpha = 0.45, colour = PAL["blue"]) +
  geom_point(data = ws, size = 0.9, colour = PAL["orange"]) +
  geom_hline(yintercept = 1, linetype = "dashed", colour = PAL["orange"], linewidth = 0.4) +
  labs(x = "median pActivity", y = "max - min among replicates",
       title = "3. replicate disagreement")
print(e1 | e2 | e3)



<p class="caption"><b>Figure 16.</b> The three anomalies. (1) The eight most frequent pActivity values, each shared by dozens of compounds. (2) Measurements reported in micromolar against those reported in nanomolar, both converted to nanomolar: the 470 micromolar-reported rows, from 83 assays, sit visibly to the right of the nanomolar ones (medians 1,200 nM against 550 nM) but overlap them heavily — a tendency, not a separation. (3) Replicate disagreement against median potency: 60 compounds (orange) disagree by more than one log unit, the worst by 3.3 — a thousandfold discrepancy between laboratories for the same molecule. Unit 6 turns this scatter into an estimate of the noise floor that caps any model's accuracy.</p>

**Anomaly 2 — the unit a paper uses carries information.** Four hundred and seventy raw measurements, from 83 assays in 77 documents, were reported in micromolar rather than nanomolar. Converted to a common unit their median is 1,200 nM against 550 nM for the nanomolar-reported rows — a factor of 2.2, with 52% of the micromolar rows above 1 µM against 40% of the nanomolar ones. The effect is modest and the two distributions overlap heavily, so this is a tendency rather than a clean separation: authors reach for micromolar when the compounds they are describing are weak. The unit is therefore not a neutral formatting choice but a weak proxy for the kind of experiment, worth recording in the data card even though it would be wrong to treat it as a label.

**Anomaly 3 — replicate disagreement.** Sixty compounds have replicate measurements differing by more than one log unit, with a median disagreement of 1.46 and an extreme case of 3.3. Three of the four worst are named, well-studied inhibitors — GSK-269962A, Y-27632 and belumosudil — measured in up to 20 documents, which is the point: the disagreement grows with the number of independent laboratories, not with the obscurity of the compound. These are not errors to delete; they are the dataset telling you how reproducible it is, and every one of them is an upper bound on how well a model can ever predict that compound.



In [ ]:
big <- names(which(table(d$scaffold_smiles) >= 5))
cl <- d |> filter(scaffold_smiles %in% big) |>
  group_by(scaffold_smiles) |>
  summarise(n = n(), spread = max(pActivity) - min(pActivity),
            weakest = min(pActivity), strongest = max(pActivity),
            mixed_types = any(n_Ki > 0) && any(n_IC50 > 0), .groups = "drop") |>
  arrange(desc(spread))
cat("scaffold families with >= 5 measured compounds:", nrow(cl),
    "| spanning more than 2 log units:", sum(cl$spread > 2), "\n")
print(as.data.frame(cl |> slice_head(n = 5) |>
  mutate(scaffold_smiles = paste0(str_sub(scaffold_smiles, 1, 32), "\u2026"),
         across(where(is.numeric), \(x) round(x, 2)))))
cat("\nof the wide families, those mixing Ki and IC50 measurements:",
    sum(cl$mixed_types[cl$spread > 2]), "\n")



A fourth observation, offered as a hypothesis rather than a result. Compounds sharing a Bemis–Murcko scaffold are structural relatives, so a family spanning more than two log units contains at least one **activity cliff** — a small structural change with a large potency consequence.<sup class="c"><a href="#r33">[33]</a></sup><sup class="c"><a href="#r34">[34]</a></sup> Thirty-four of the 94 families with at least five measured compounds qualify, the widest spanning 4.53 log units over only ten compounds. Such compounds are the most informative in a series and the ones that break similarity-based models. They also have a mundane alternative explanation that has to be excluded first: six of these wide families mix Ki and IC50 measurements, and pooling those two quantities is itself a documented source of roughly half a log unit of noise.<sup class="c"><a href="#r30">[30]</a></sup> Telling a real cliff from a measurement artefact needs the structures themselves; the EDA's job is to find the candidates and label them as candidates.

<div class="box bridge"><h4>Bridge</h4>
<p><strong>Backwards:</strong> Unit 4 (<a href="../U4_Tidy_Data/index.html">Tidy data in the tidyverse</a>) built the compound table explored here, and Unit 3 downloaded it (<a href="../U3_Getting_Data/index.html">Python edition</a>). <strong>Forwards:</strong> Unit 6 (<a href="../U6_Data_Quality/index.html">Data quality, curation and bias</a>) turns the anomalies of Section 11 into quantitative bias diagnostics and a noise floor; Unit 7 attaches uncertainty to the summaries drawn here (<a href="../U7_Estimation/index.html">Python edition</a>); Unit 20 applies the same EDA discipline to time series, where autocorrelation makes every plot in this unit misleading in a new way (<a href="../U20_MD_Analytics/index.html">Python edition</a>); and Unit 23 takes these figures to publication quality (<a href="../U23_Communication/index.html">Python edition</a>). The <a href="../index.html">R-edition index</a> lists every unit and the <a href="../../Python/U5_EDA/index.html">Python edition of this unit</a> does the same analysis with matplotlib and seaborn. <strong>Sister courses:</strong> the chemical meaning of the descriptors plotted here is developed in <a href="../../../Cheminformatics/U5_Descriptors_Fingerprints/index.html">Unit 5</a> and the scaffold and chemical-space views in <a href="../../../Cheminformatics/U8_Chemical_Space/index.html">Unit 8</a> of the cheminformatics course; activity cliffs are the subject of its <a href="../../../Cheminformatics/U11_Similarity_SAR/index.html">Unit 11</a>; and the R edition of the AI course covers the modelling that follows, beginning with <a href="../../../BGM411/R/S0_R_Toolkit/index.html">its R toolkit</a>.</p></div>

## 12. Summary

- **EDA is a checklist**, not an inspiration: shape and grain, missingness, univariate distributions, pairwise relations, group comparisons, anomalies, write-up. Run it before any model.
- **The grammar of graphics** — data, aesthetics, geometry, statistic, scale, facet — turns "which chart?" into a set of explicit choices, and in R `ggplot2` is that grammar rather than an analogy to it.
- **Every distribution display has a hidden parameter** except the ECDF: bin width for histograms (Scott 0.28, Freedman–Diaconis 0.22 log units here), bandwidth for density estimates, which also leak density past the data's boundary.
- **Box plots answer one question.** Two samples with matched quartiles had entirely different shapes and excess kurtosis −0.01 against −1.22. Draw the points whenever the group is smaller than a few hundred.
- **Overplotting is a measurement problem.** At 4,280 points an opaque scatter shows only the silhouette; transparency, 2-D binning (busiest cell: 94 compounds) and subsampling each trade something, and the caption must say which.
- **Correlation matrices show redundancy, not causation.** `MolWt` and `HeavyAtoms` correlate at 0.979; no descriptor correlates with potency above 0.241.
- **Anscombe's quartet and the Datasaurus construction** are proofs, not anecdotes: three of our generated datasets share five summary statistics to two decimals and look nothing alike.
- **Colour is data encoding.** Perceptually uniform maps for quantities, Okabe–Ito for categories, and simulate a colour-vision deficiency before you publish — the ad-hoc palette's closest pair fell from an RGB distance of 0.23 to 0.08 under deuteranopia, while Okabe–Ito held at 0.20.
- **Three real anomalies** in this dataset: value pile-ups at censoring limits and round numbers, including 141 compounds reported at exactly 550 nM by four assays; 470 measurements from 83 assays reported in micromolar, whose median is 2.2 times weaker than the nanomolar-reported ones; and 60 compounds whose replicates disagree by more than a log unit, the worst by 3.3.

## 13. Self-assessment

<details><summary>1. What is the one question EDA must answer before any model is fitted?</summary>
<p>What is actually in the table — its grain, its gaps, the shape of each variable and the relations between them — so that you know which modelling assumptions are already violated. The secondary purpose is to generate hypotheses; those must then be tested on data not used to find them.</p></details>

<details><summary>2. Name the layers of the grammar of graphics and map a 2-D binned density plot onto them.</summary>
<p>Data, aesthetic mapping, geometry, statistical transformation, scales, facets, coordinates/theme. The binned plot is: data = the compound table; aesthetics x = MolWt, y = MolLogP, fill = count; geometry = tile; statistic = count of points per rectangular bin; scales = linear axes and a sequential colour map; no facets. In <code>ggplot2</code> each of those is a separate function call in the same expression.</p></details>

<details><summary>3. Why is the ECDF the only distribution display with no tuning parameter, and why is it still not the default?</summary>
<p>Because it plots the data themselves: for each observed value, the fraction of observations at or below it. Nothing is binned or smoothed. It is not the default because most readers cannot see a mode in it — the eye reads density far better from a histogram than slope from a cumulative curve.</p></details>

<details><summary>4. Your density curve shows mass below the smallest observed value. What happened and what do you do?</summary>
<p>The kernel is symmetric and spills past the boundary, assigning density to impossible values. Either state the boundary explicitly and truncate, use a boundary-corrected or reflected kernel, or show an ECDF or histogram instead. Never present the spill as if it were data.</p></details>

<details><summary>5. A referee asks you to replace your strip plot of six groups with box plots. When should you refuse?</summary>
<p>When the groups are small or unequal, or when any distribution is multimodal or truncated. In Figure 6 family S6 has 16 compounds and S2 is a wall of 23 points against a ceiling at 5.10 — a box plot shows neither. Offer a box overlaid with the points, which satisfies both of you.</p></details>

<details><summary>6. Two descriptors correlate at 0.98. What does that mean for an EDA report and for a model?</summary>
<p>They carry one piece of information between them. In the report, say so and show one. In a model, the pair makes a linear fit's coefficients unstable and uninterpretable (collinearity), though it rarely hurts a tree ensemble's predictions; feature selection or a dimensionality reduction (Unit 13) is the standard response.</p></details>

<details><summary>7. What exactly does Anscombe's quartet prove?</summary>
<p>That mean, variance, correlation and the least-squares line are compatible with structurally different data — a linear relation with noise, a parabola, a line plus an outlier, and a vertical stack plus one leverage point. Numerical summaries are necessary and never sufficient; the plot is part of the analysis, not the presentation.</p></details>

<details><summary>8. Why is a truncated y axis dishonest on a bar chart but acceptable on a line chart?</summary>
<p>Because a bar encodes the value as a <em>length</em> measured from the baseline, so moving the baseline changes the encoded quantity; truncating at 500 made our largest class look 15.2 times the smallest instead of 2.9. A line encodes value as position and the reader reads changes rather than lengths, so a restricted range is a legitimate zoom — provided the axis is labelled.</p></details>

<details><summary>9. Why is <code>jet</code> a bad colour map and <code>viridis</code> a good one?</summary>
<p>Perceived lightness along <code>jet</code> rises, falls and rises again, so equal data steps are not equal perceptual steps: false boundaries appear at the yellow and cyan bands and real contrast is lost in the green. <code>viridis</code> is constructed to increase monotonically in lightness and to remain distinguishable under the common colour-vision deficiencies, so it also survives greyscale printing.</p></details>

<details><summary>10. You find 141 compounds with exactly the same reported potency. What do you do?</summary>
<p>Trace them to their source records before anything else — here, one assay reporting a single representative value for 124 compounds of a series. Then record the finding in the data card, treat those compounds as carrying far less information than their count suggests, and make sure the train/test split does not place near-duplicates of the same value on both sides.</p></details>

<details><summary>11. Why is "measurements reported in µM are the weak ones" a problem rather than a curiosity?</summary>
<p>Because the unit becomes a weak proxy for the experiment type: the micromolar-reported rows are 2.2 times weaker on median than the nanomolar ones. The shift is modest and the distributions overlap, so it is not a label to split on, but it means the reported unit is correlated with the answer — something to record in the data card and to check before treating the pooled set as homogeneous.</p></details>

<details><summary>12. How many of the plots in this unit would you put in a paper, and what changes first?</summary>
<p>Very few, and that is the point: EDA figures are for you. For publication the changes are a single clear message per figure, axis labels with units, a colour-blind-safe palette, the sample size stated in the panel or caption, and the caption reporting the numbers the reader is meant to take away. Unit 23 does this systematically.</p></details>

## Glossary

<dl class="gloss">
<dt>Aesthetic mapping</dt><dd>The assignment of data columns to visual channels — position, colour, size, shape, transparency; <code>aes()</code> in ggplot2.</dd>
<dt>Anti-pattern</dt><dd>A chart convention that reliably misleads: truncated bar axis, dual axis, rainbow colour map, pie with many slices.</dd>
<dt>Bandwidth</dt><dd>The width of the kernel in a density estimate; the equivalent of bin width, and its hidden parameter (<code>adjust</code> in R's <code>density()</code>).</dd>
<dt>Bemis–Murcko scaffold</dt><dd>A molecule reduced to its ring systems and the linkers between them; the standard way of grouping compounds into chemical families.</dd>
<dt>Clustered heatmap</dt><dd>A heatmap whose rows and columns are reordered by hierarchical clustering so that correlated blocks become contiguous.</dd>
<dt>Deuteranopia</dt><dd>The commonest colour-vision deficiency, a reduced ability to distinguish red from green; simulated here with the Machado model.</dd>
<dt>ECDF</dt><dd>Empirical cumulative distribution function: for each value, the fraction of observations at or below it. No tuning parameter.</dd>
<dt>Exploratory data analysis</dt><dd>Examining data to find structure and generate hypotheses, as distinct from confirmatory analysis, which tests a hypothesis formed beforehand.</dd>
<dt>Grammar of graphics</dt><dd>The decomposition of a statistical graphic into data, aesthetics, geometry, statistical transformation, scales, facets and coordinates.</dd>
<dt>Overplotting</dt><dd>So many marks that the ink saturates and density becomes unreadable; treated with transparency, binning or subsampling.</dd>
<dt>Pile-up</dt><dd>A value shared by far more records than chance allows, usually a reporting convention rather than a measurement.</dd>
<dt>Small multiples</dt><dd>One small plot repeated across the levels of a variable with shared axes, so the panels are directly comparable; <code>facet_wrap()</code>.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Wickham H. A layered grammar of graphics. <i>Journal of Computational and Graphical Statistics</i> 2010;19(1):3–28. doi:<a href="https://doi.org/10.1198/jcgs.2009.07098">10.1198/jcgs.2009.07098</a></li>
<li id="r2">Hunter JD. Matplotlib: a 2D graphics environment. <i>Computing in Science &amp; Engineering</i> 2007;9(3):90–95. doi:<a href="https://doi.org/10.1109/MCSE.2007.55">10.1109/MCSE.2007.55</a></li>
<li id="r3">Waskom ML. seaborn: statistical data visualization. <i>Journal of Open Source Software</i> 2021;6(60):3021. doi:<a href="https://doi.org/10.21105/joss.03021">10.21105/joss.03021</a></li>
<li id="r4">Harris CR, Millman KJ, van der Walt SJ, <i>et al.</i> Array programming with NumPy. <i>Nature</i> 2020;585:357–362. doi:<a href="https://doi.org/10.1038/s41586-020-2649-2">10.1038/s41586-020-2649-2</a></li>
<li id="r5">Virtanen P, Gommers R, Oliphant TE, <i>et al.</i> SciPy 1.0: fundamental algorithms for scientific computing in Python. <i>Nature Methods</i> 2020;17:261–272. doi:<a href="https://doi.org/10.1038/s41592-019-0686-2">10.1038/s41592-019-0686-2</a></li>
<li id="r6">McKinney W. Data structures for statistical computing in Python. <i>Proceedings of the 9th Python in Science Conference</i> 2010:56–61. doi:<a href="https://doi.org/10.25080/Majora-92bf1922-00a">10.25080/Majora-92bf1922-00a</a></li>
<li id="r7">Wickham H. Tidy data. <i>Journal of Statistical Software</i> 2014;59(10):1–23. doi:<a href="https://doi.org/10.18637/jss.v059.i10">10.18637/jss.v059.i10</a></li>
<li id="r8">Rougier NP, Droettboom M, Bourne PE. Ten simple rules for better figures. <i>PLoS Computational Biology</i> 2014;10(9):e1003833. doi:<a href="https://doi.org/10.1371/journal.pcbi.1003833">10.1371/journal.pcbi.1003833</a></li>
<li id="r9">Anscombe FJ. Graphs in statistical analysis. <i>The American Statistician</i> 1973;27(1):17–21. doi:<a href="https://doi.org/10.1080/00031305.1973.10478966">10.1080/00031305.1973.10478966</a></li>
<li id="r10">Matejka J, Fitzmaurice G. Same stats, different graphs: generating datasets with varied appearance and identical statistics through simulated annealing. <i>Proceedings of the 2017 CHI Conference on Human Factors in Computing Systems</i> 2017:1290–1294. doi:<a href="https://doi.org/10.1145/3025453.3025912">10.1145/3025453.3025912</a></li>
<li id="r11">Cleveland WS, McGill R. Graphical perception: theory, experimentation, and application to the development of graphical methods. <i>Journal of the American Statistical Association</i> 1984;79(387):531–554. doi:<a href="https://doi.org/10.1080/01621459.1984.10478080">10.1080/01621459.1984.10478080</a></li>
<li id="r12">Scott DW. On optimal and data-based histograms. <i>Biometrika</i> 1979;66(3):605–610. doi:<a href="https://doi.org/10.1093/biomet/66.3.605">10.1093/biomet/66.3.605</a></li>
<li id="r13">Freedman D, Diaconis P. On the histogram as a density estimator: L2 theory. <i>Zeitschrift für Wahrscheinlichkeitstheorie und verwandte Gebiete</i> 1981;57(4):453–476. doi:<a href="https://doi.org/10.1007/BF01025868">10.1007/BF01025868</a></li>
<li id="r14">Wand MP. Data-based choice of histogram bin width. <i>The American Statistician</i> 1997;51(1):59–64. doi:<a href="https://doi.org/10.1080/00031305.1997.10473591">10.1080/00031305.1997.10473591</a></li>
<li id="r15">Hintze JL, Nelson RD. Violin plots: a box plot–density trace synergism. <i>The American Statistician</i> 1998;52(2):181–184. doi:<a href="https://doi.org/10.1080/00031305.1998.10480559">10.1080/00031305.1998.10480559</a></li>
<li id="r16">McGill R, Tukey JW, Larsen WA. Variations of box plots. <i>The American Statistician</i> 1978;32(1):12–16. doi:<a href="https://doi.org/10.1080/00031305.1978.10479236">10.1080/00031305.1978.10479236</a></li>
<li id="r17">Krzywinski M, Altman N. Visualizing samples with box plots. <i>Nature Methods</i> 2014;11:119–120. doi:<a href="https://doi.org/10.1038/nmeth.2813">10.1038/nmeth.2813</a></li>
<li id="r18">Streit M, Gehlenborg N. Bar charts and box plots. <i>Nature Methods</i> 2014;11:117. doi:<a href="https://doi.org/10.1038/nmeth.2807">10.1038/nmeth.2807</a></li>
<li id="r19">Weissgerber TL, Milic NM, Winham SJ, Garovic VD. Beyond bar and line graphs: time for a new data presentation paradigm. <i>PLoS Biology</i> 2015;13(4):e1002128. doi:<a href="https://doi.org/10.1371/journal.pbio.1002128">10.1371/journal.pbio.1002128</a></li>
<li id="r20">Weissgerber TL, Winham SJ, Heinzen EP, <i>et al.</i> Reveal, don't conceal: transforming data visualization to improve transparency. <i>Circulation</i> 2019;140(18):1506–1518. doi:<a href="https://doi.org/10.1161/CIRCULATIONAHA.118.037777">10.1161/CIRCULATIONAHA.118.037777</a></li>
<li id="r21">Allen M, Poggiali D, Whitaker K, Marshall TR, van Langen J, Kievit RA. Raincloud plots: a multi-platform tool for robust data visualization. <i>Wellcome Open Research</i> 2021;4:63. doi:<a href="https://doi.org/10.12688/wellcomeopenres.15191.2">10.12688/wellcomeopenres.15191.2</a></li>
<li id="r22">Hartigan JA, Hartigan PM. The dip test of unimodality. <i>The Annals of Statistics</i> 1985;13(1):70–84. doi:<a href="https://doi.org/10.1214/aos/1176346577">10.1214/aos/1176346577</a></li>
<li id="r23">Crameri F, Shephard GE, Heron PJ. The misuse of colour in science communication. <i>Nature Communications</i> 2020;11:5444. doi:<a href="https://doi.org/10.1038/s41467-020-19160-7">10.1038/s41467-020-19160-7</a></li>
<li id="r24">Borland D, Taylor RM. Rainbow color map (still) considered harmful. <i>IEEE Computer Graphics and Applications</i> 2007;27(2):14–17. doi:<a href="https://doi.org/10.1109/MCG.2007.323435">10.1109/MCG.2007.323435</a></li>
<li id="r25">Machado GM, Oliveira MM, Fernandes LAF. A physiologically-based model for simulation of color vision deficiency. <i>IEEE Transactions on Visualization and Computer Graphics</i> 2009;15(6):1291–1298. doi:<a href="https://doi.org/10.1109/TVCG.2009.113">10.1109/TVCG.2009.113</a></li>
<li id="r26">Harrower M, Brewer CA. ColorBrewer.org: an online tool for selecting colour schemes for maps. <i>The Cartographic Journal</i> 2003;40(1):27–37. doi:<a href="https://doi.org/10.1179/000870403235002042">10.1179/000870403235002042</a></li>
<li id="r27">Nuñez JR, Anderton CR, Renslow RS. Optimizing colormaps with consideration for color vision deficiency to enable accurate interpretation of scientific data. <i>PLoS ONE</i> 2018;13(7):e0199239. doi:<a href="https://doi.org/10.1371/journal.pone.0199239">10.1371/journal.pone.0199239</a></li>
<li id="r28">Gelman A, Unwin A. Infovis and statistical graphics: different goals, different looks. <i>Journal of Computational and Graphical Statistics</i> 2013;22(1):2–28. doi:<a href="https://doi.org/10.1080/10618600.2012.761137">10.1080/10618600.2012.761137</a></li>
<li id="r29">Zdrazil B, Felix E, Hunter F, <i>et al.</i> The ChEMBL Database in 2023. <i>Nucleic Acids Research</i> 2024;52(D1):D1180–D1192. doi:<a href="https://doi.org/10.1093/nar/gkad1004">10.1093/nar/gkad1004</a></li>
<li id="r30">Kalliokoski T, Kramer C, Vulpetti A, Gedeck P. Comparability of mixed IC50 data — a statistical analysis. <i>PLoS ONE</i> 2013;8(4):e61007. doi:<a href="https://doi.org/10.1371/journal.pone.0061007">10.1371/journal.pone.0061007</a></li>
<li id="r31">Lipinski CA, Lombardo F, Dominy BW, Feeney PJ. Experimental and computational approaches to estimate solubility and permeability in drug discovery and development settings. <i>Advanced Drug Delivery Reviews</i> 2001;46(1–3):3–26. doi:<a href="https://doi.org/10.1016/S0169-409X(00)00129-0">10.1016/S0169-409X(00)00129-0</a></li>
<li id="r32">Bickerton GR, Paolini GV, Besnard J, Muresan S, Hopkins AL. Quantifying the chemical beauty of drugs. <i>Nature Chemistry</i> 2012;4:90–98. doi:<a href="https://doi.org/10.1038/nchem.1243">10.1038/nchem.1243</a></li>
<li id="r33">Maggiora GM. On outliers and activity cliffs — why QSAR often disappoints. <i>Journal of Chemical Information and Modeling</i> 2006;46(4):1535. doi:<a href="https://doi.org/10.1021/ci060117s">10.1021/ci060117s</a></li>
<li id="r34">Stumpfe D, Bajorath J. Exploring activity cliffs in medicinal chemistry. <i>Journal of Medicinal Chemistry</i> 2012;55(7):2932–2942. doi:<a href="https://doi.org/10.1021/jm201706b">10.1021/jm201706b</a></li>
<li id="r35">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <i>Journal of Medicinal Chemistry</i> 1996;39(15):2887–2893. doi:<a href="https://doi.org/10.1021/jm9602928">10.1021/jm9602928</a></li>
<li id="r36">Wickham H, Averick M, Bryan J, <i>et al.</i> Welcome to the Tidyverse. <i>Journal of Open Source Software</i> 2019;4(43):1686. doi:<a href="https://doi.org/10.21105/joss.01686">10.21105/joss.01686</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Tukey JW. <i>Exploratory Data Analysis</i>. Addison-Wesley, 1977 — the book that named the subject.</li>
<li>Wilkinson L. <i>The Grammar of Graphics</i>, 2nd ed. Springer, 2005 — the formal source of Section 2.</li>
<li>Wickham H, Navarro D, Pedersen TL. <i>ggplot2: Elegant Graphics for Data Analysis</i>, 3rd ed. — free online; the reference for every figure here.</li>
<li>Wilke CO. <i>Fundamentals of Data Visualization</i>. O'Reilly, 2019 — free online; written with R, applicable anywhere.</li>
<li>Healy K. <i>Data Visualization: A Practical Introduction</i>. Princeton University Press, 2018 — free online, all examples in R.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>The ggplot2 reference and the "Layered grammar" vignette; the patchwork vignette for multi-panel composition.</li>
<li><code>corrplot</code> introduction vignette; <code>pheatmap</code> reference manual.</li>
<li>Scientific colour maps (Crameri), the Okabe–Ito palette definition and R's own <code>palette.colors()</code>.</li>
<li>From Data to Viz (data-to-viz.com) — a decision tree from data shape to chart type, with the caveats of each and R code for all of them.</li>
</ul>
